# HydroFlow Foods — full YOLOv8m training (Google Drive)

| Item | Value |
|------|--------|
| Local path | `D:\Ai_Project\HydroFlow_V2\data\processed\object_detect\foods` |
| Drive path | **Computers → My Laptop →** `Ai_Project/HydroFlow_V2/data/processed/object_detect/foods` |
| Classes | `rice`, `oil`, `canned_fish` |
| Model | YOLOv8m |

1. **Runtime → Change runtime type → T4 GPU**
2. Run cells **1 → 8** in order
3. After sync, weights on PC: `D:\Ai_Project\HydroFlow_V2\ml\models\object_detect\foods\best.pt`

## Cell 1 — Install

In [ ]:
!pip install -q ultralytics

## Cell 2 — Mount Drive & locate dataset

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path("/content/drive")
FOODS_REL = Path("Ai_Project/HydroFlow_V2/data/processed/object_detect/foods")

FOODS_CANDIDATES = [
    DRIVE / "Other computers" / "My laptop" / FOODS_REL,
    DRIVE / "Other computers" / "My Laptop" / FOODS_REL,
    DRIVE / "Computers" / "My laptop" / FOODS_REL,
    DRIVE / "Computers" / "My Laptop" / FOODS_REL,
    DRIVE / "MyDrive" / FOODS_REL,
]


def is_foods_dataset(root: Path) -> bool:
    return (
        (root / "data.yaml").is_file()
        and (root / "train" / "images").is_dir()
        and (root / "train" / "labels").is_dir()
        and (root / "valid" / "images").is_dir()
        and (root / "valid" / "labels").is_dir()
    )


def find_foods_dir() -> Path:
    for path in FOODS_CANDIDATES:
        if is_foods_dataset(path):
            return path
    matches = [
        p.parent
        for p in DRIVE.rglob("data.yaml")
        if p.parent.name == "foods" and "object_detect" in p.as_posix()
    ]
    for path in matches:
        if is_foods_dataset(path):
            return path
    raise FileNotFoundError(
        "foods dataset not found. Run in a new cell:\n"
        '!find /content/drive -path "*/object_detect/foods/data.yaml" 2>/dev/null'
    )


FOODS_DIR = find_foods_dir()
REPO_ROOT = FOODS_DIR.parents[4]

print("FOODS_DIR =", FOODS_DIR)
print("REPO_ROOT =", REPO_ROOT)
print("Train images:", len(list((FOODS_DIR / "train/images").glob("*.jpg"))))
print("Valid images:", len(list((FOODS_DIR / "valid/images").glob("*.jpg"))))

## Cell 3 — Patch `data.yaml` & copy dataset to Colab disk

In [ ]:
def write_data_yaml(yaml_path: Path, dataset_root: Path) -> None:
    lines = []
    for line in yaml_path.read_text(encoding="utf-8").splitlines():
        if line.startswith("path:"):
            lines.append(f"path: {dataset_root.as_posix()}")
        else:
            lines.append(line)
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


LOCAL_FOODS = Path("/content/foods_dataset")
if LOCAL_FOODS.exists():
    shutil.rmtree(LOCAL_FOODS)
shutil.copytree(FOODS_DIR, LOCAL_FOODS)
write_data_yaml(LOCAL_FOODS / "data.yaml", LOCAL_FOODS)

TRAIN_DATA = str(LOCAL_FOODS / "data.yaml")
print((LOCAL_FOODS / "data.yaml").read_text())
print("\nUsing:", TRAIN_DATA)

## Cell 4 — Train YOLOv8m

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
IMGSZ = 640
BATCH = 16
PATIENCE = 20
RUN_NAME = "foods_yolov8m_colab"

RUNS_DIR = REPO_ROOT / "ml" / "runs" / "yolo"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO("yolov8m.pt")
model.train(
    data=TRAIN_DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    pretrained=True,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    val=True,
    save=True,
)

BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
LAST_PT = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
print("best.pt:", BEST_PT)
print("last.pt:", LAST_PT)

## Cell 5 — Validation metrics & training curves

In [ ]:
from IPython.display import Image, display

results_png = RUNS_DIR / RUN_NAME / "results.png"
confusion_png = RUNS_DIR / RUN_NAME / "confusion_matrix_normalized.png"

if results_png.is_file():
    display(Image(filename=str(results_png), width=900))
if confusion_png.is_file():
    display(Image(filename=str(confusion_png), width=700))

val_model = YOLO(str(BEST_PT))
metrics = val_model.val(data=TRAIN_DATA, imgsz=IMGSZ, device=0)
print(metrics)

## Cell 6 — Test predictions (sample val images)

In [ ]:
import random

PREVIEW_DIR = Path("/content/preview_preds")
if PREVIEW_DIR.exists():
    shutil.rmtree(PREVIEW_DIR)
PREVIEW_DIR.mkdir(parents=True)

val_images = list((LOCAL_FOODS / "valid/images").glob("*.jpg"))
samples = random.sample(val_images, min(6, len(val_images)))

pred = YOLO(str(BEST_PT))
pred.predict(
    source=[str(p) for p in samples],
    imgsz=IMGSZ,
    conf=0.35,
    save=True,
    project=str(PREVIEW_DIR),
    name="val_samples",
    exist_ok=True,
)

saved = list((PREVIEW_DIR / "val_samples").glob("*.jpg"))
for img in saved[:6]:
    display(Image(filename=str(img), width=480))

## Cell 7 — Copy weights into HydroFlow (Drive → syncs to laptop)

In [ ]:
SERVED_PT = REPO_ROOT / "ml" / "models" / "object_detect" / "foods" / "best.pt"
BACKUP_PT = FOODS_DIR / "trained_weights" / "best.pt"

SERVED_PT.parent.mkdir(parents=True, exist_ok=True)
BACKUP_PT.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_PT, SERVED_PT)
shutil.copy2(BEST_PT, BACKUP_PT)

print("App weights (sync to PC):", SERVED_PT)
print("Backup on Drive:", BACKUP_PT)

## Cell 8 — Browser download (optional)

In [ ]:
from google.colab import files

files.download(str(BEST_PT))

---

### Troubleshooting

- **CUDA out of memory:** set `BATCH = 8` in Cell 4 and re-run from Cell 4.
- **Dataset not found:** run `!find /content/drive -path "*/object_detect/foods/data.yaml" 2>/dev/null` and set `FOODS_DIR = Path("...")` manually in Cell 2.
- **App still wrong model:** remove or rename `ml/models/object_detect/v2/best.pt` if you only want the foods model.

### One-shot cell (alternative)

If you prefer a single cell, use **Cell 9** below instead of Cells 2–7.

## Cell 9 — ALL-IN-ONE (mount → train → export)

In [ ]:
# Run ONLY this cell after Cell 1 (pip install), if you want everything at once.

from __future__ import annotations

import shutil
from pathlib import Path

from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/drive")

DRIVE = Path("/content/drive")
REL = Path("Ai_Project/HydroFlow_V2/data/processed/object_detect/foods")
CANDS = [
    DRIVE / "Other computers" / "My laptop" / REL,
    DRIVE / "Other computers" / "My Laptop" / REL,
    DRIVE / "Computers" / "My Laptop" / REL,
    DRIVE / "MyDrive" / REL,
]

def ok(p: Path) -> bool:
    return (p / "data.yaml").is_file() and (p / "train/images").is_dir()

FOODS_DIR = next(p for p in CANDS if ok(p))
REPO = FOODS_DIR.parents[4]
LOCAL = Path("/content/foods_dataset")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(FOODS_DIR, LOCAL)

yaml = LOCAL / "data.yaml"
yaml.write_text(
    "path: " + LOCAL.as_posix() + "\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    "nc: 3\n"
    "names: [rice, oil, canned_fish]\n",
    encoding="utf-8",
)

RUNS = REPO / "ml" / "runs" / "yolo"
RUNS.mkdir(parents=True, exist_ok=True)
NAME = "foods_yolov8m_colab"

YOLO("yolov8m.pt").train(
    data=str(yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,
    close_mosaic=10,
    project=str(RUNS),
    name=NAME,
    exist_ok=True,
    plots=True,
    val=True,
)

BEST = RUNS / NAME / "weights" / "best.pt"
OUT = REPO / "ml" / "models" / "object_detect" / "foods" / "best.pt"
OUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, OUT)
shutil.copy2(BEST, FOODS_DIR / "trained_weights" / "best.pt")

print("FOODS_DIR:", FOODS_DIR)
print("best.pt:", BEST)
print("HydroFlow served:", OUT)
print(YOLO(str(BEST)).val(data=str(yaml), imgsz=640, device=0))